In [1]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:
  Successfully uninstalled matplotlib-3.10.0
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import warnings
import subprocess
warnings.simplefilter('ignore')

In [3]:
def set_env_tar(input_archive, temp_dir):
    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)


def set_env(input_dir):    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{input_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)

In [4]:
set_env_tar(input_archive='/kaggle/input/notebooks/andreasbis/aimo-3-utils/wheels.tar.gz', temp_dir='/kaggle/tmp/setup')

Looking in links: /kaggle/tmp/setup/wheels
Processing /kaggle/tmp/setup/wheels/unsloth-2025.12.9-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/trl-0.24.0-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/vllm-0.11.2-cp38-abi3-manylinux1_x86_64.whl
Processing /kaggle/tmp/setup/wheels/openai_harmony-0.0.8-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/tmp/setup/wheels/unsloth_zoo-2025.12.7-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/tyro-1.0.3-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/xformers-0.0.33.post1-cp39-abi3-manylinux_2_28_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/bitsandbytes-0.49.0-py3-none-manylinux_2_24_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/datasets-4.3.0-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/transformers-4.57.3-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/prometheus_fastapi_instrumentator-7.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyldavis 3.4.1 requires scikit-learn>=1.0.0, which is not installed.
ydata-profiling 4.18.1 requires matplotlib<=3.10,>=3.5, which is not installed.
librosa 0.11.0 requires scikit-learn>=1.1.0, which is not installed.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.31.0 requires matplotlib>=3.7.1, which is not installed.
shap 0.50.0 requires scikit-learn, which is not installed.
pynndescent 0.6.0 requires scikit-learn>=0.18, which is not installed.
umap-learn 0.5.11 requires scikit-learn>=1.6, which is not installed.
sentence-transformers 5.2.0 requires scikit-learn, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
arviz 0.22.0 requires matplotlib>=3.8, which is not installed.
cuml-cu12 25.10

In [5]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'
#os.environ['VLLM_MXFP4_USE_MARLIN'] = '1'

In [6]:
import gc
import re
import copy
import math
import time
import queue
import random
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI
from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [7]:
class CFG:           
    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1'

    dtype = 'auto'
    kv_cache_dtype = 'fp8_e4m3'
    calculate_kv_scales = False
    
    max_absolute_timeout = 680
    min_absolute_timeout = 200

    notebook_limit = 17525
    server_timeout = 300

    session_timeout = 150
    jupyter_timeout = 16  # Default 6 - Time to wait a code execution
    sandbox_timeout = 3   # Default 3 - Time to wait a sandbox from the queue

    stream_interval = 50 
    context_tokens = 80000 #65536
    buffer_tokens = 256
    search_tokens = 48
    attempts = 6
    top_logprobs = 12
    max_num_seqs = attempts 
    max_num_batched_tokens = 8192
    max_cudagraph_capture_size = 2048
    workers = 8            
    turns = 150                        
    seed = 11

    gpu_memory_utilization = 0.96
    min_p = 0.065
    top_p = 1.0
    top_k = 100
    base_temperature_phase_1 = 0.65
    base_temperature_phase_2 = 0.70
    
    window_entropy = 300
    entropy_threshold = 1.26

    early_stop_phase_1 = None  
    early_stop_phase_2 = 2 

    system_prompt = """
    You are an International Mathematical Olympiad (IMO) Competitor. 
    
    IMO problems require creative thinking. Here are some strategies:
    - Break the problem into smaller parts: Simplify complex questions by solving smaller sections first.
    - Try different methods: Experiment with algebraic, geometric, and number theory approaches or mix them.
    - Look for patterns: Many problems follow a pattern. Identifying it can make solving easier.
    - Check your work: Avoid careless mistakes by double-checking calculations and the final answer.
    - Retrace your steps and be willing to start over with a different approach or strategy.
    - Use Python to support your reasoning, making sure the code is executable and error-free. Use fast libraries to speed up the calculation.

    The final answer is a non-negative integer between 0 and 99999 inclusive. 
    Please reason step by step, and put your final answer within \\boxed{}.
    """
    
    tool_prompt = """
    Use this tool to execute Python code in your chain of thought. The code will not be shown to the user. This tool should be used for internal reasoning, but not for code that is intended to be visible to the user (e.g. when creating plots, tables, or files).
    When you send a message containing Python code to python, it will be executed in a stateful Jupyter notebook environment. You have to use print statements to access the output.
    """  

    phase_2_extra_prompt = """
    This is a quick assessment phase. Your goal is to reach a confident answer as fast as possible.
    - Avoid long explorations: if you have a clear approach, execute it directly.
    - Use Python immediately for any calculation rather than doing it by hand.
    - Skip re-explaining the problem or restating what is given.
    - Do not explore multiple methods: commit to the most promising one and follow it through.
    - As soon as you have a confident answer, write \\boxed{answer} and stop.
    """

In [8]:
set_seed(CFG.seed)

In [9]:
class AIMO3Template:
    def __init__(self):
        pass

    def get_system_content(
        self, 
        system_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> SystemContent:
        
        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    
    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:
        
        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)
        user_message = Message.from_role_and_content(Role.USER, user_prompt)
        return [system_message, user_message]

In [10]:
class AIMO3Sandbox:
    _port_lock = threading.Lock()
    def __init__(self, timeout: float):
        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
    
        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'
    
        self._km = KernelManager()
    
        # Minimal retry logic in case the kernel fails to start
        attempts = 3
        for attempt in range(1, attempts + 1):
            try:
                self._km.start_kernel(
                    env=env,
                    extra_arguments=['--Application.log_level=CRITICAL']
                )
                break
            except Exception:
                try:
                    self._km.cleanup_resources()
                except Exception:
                    pass
    
                if attempt == attempts:
                    raise
    
                # Small backoff before retrying
                time.sleep(0.1 * attempt)
    
        # Create client and wait for kernel to become ready
        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True
    
        # Preload commonly used math/scientific libraries inside the sandbox
        self.toolbox_imports = """
        import bisect
        import collections
        import functools
        import heapq
        import itertools
        import math
        import operator
        import random
        import os
        import re
        import sys
        import time
        from collections import Counter, defaultdict, deque
        from decimal import Decimal, getcontext
        from fractions import Fraction
        
        import mpmath as mp
        import numpy as np
        import sympy as sp    
        
        mp.mp.dps = 64
        sys.set_int_max_str_digits(0)
        getcontext().prec = 64
        """
        
        # Execute toolbox imports inside the kernel session
        self.execute(self.toolbox_imports)

    
    def _format_error(self, traceback: list[str]) -> str:
        clean_lines = []
        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    
    def execute(self, code: str, timeout: float | None = None) -> str:
        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()
                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=0.5)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])
                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')
                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else f'[WARN] No output available. Use print() to output anything to stdout to receive the output'

    
    def close(self):
        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    
    def reset(self):
        self.execute('%reset -f\n' + self.toolbox_imports)

    
    def __del__(self):
        self.close()

In [11]:
class AIMO3Tool:
    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):
        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    
    def _ensure_session(self):
        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    
    def _ensure_last_print(self, code: str) -> str:
        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    
    @property
    def instruction(self) -> str:
        return self._tool_prompt

    
    @property
    def tool_config(self) -> ToolNamespaceConfig:
        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    
    def _make_response(self, output: str, channel: str | None = None) -> Message:
        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')
        if channel:
            message = message.with_channel(channel)
        return message

    
    def process_sync_plus(self, message: Message) -> list[Message]:
        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)
        
        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'
                
        return [self._make_response(output, channel=message.channel)]

In [12]:
class AIMO3Solver:
    def __init__(self, cfg, port: int = 8000, notebook_start_time: int = 0):
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()

        self.phase_1_attempts = self.cfg.attempts // 2
        self.phase_2_attempts = self.cfg.attempts - self.phase_1_attempts
        
        self.temperatures_phase_1 = [self.cfg.base_temperature_phase_1 + 0.05 * i for i in range(self.phase_1_attempts)]
        self.temperatures_phase_2 = [self.cfg.base_temperature_phase_2 + 0.10 * i for i in range(self.phase_2_attempts)]

        self.max_num_seqs = max(self.phase_1_attempts, self.phase_2_attempts)
    
        self._preload_model_weights()
        
        self.server_process = self._start_server()
    
        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )
        
        self._wait_for_server()
        self._initialize_kernels()
    
        self.notebook_start_time = notebook_start_time or time.time()
        self.problems_remaining = 50

        self.extra_body = {
            "top_k": self.cfg.top_k,
            "min_p": self.cfg.min_p,
            "top_p": self.cfg.top_p,
            "stop_token_ids": self.stop_token_ids,
            "return_token_ids": True,
        }

        #self.boxed_pattern = re.compile(r'\\boxed\s*\{\s*([0-9][0-9,]{0,10})\s*\}')
        self.boxed_pattern = re.compile(r'\\{1,2}boxed\s*\{\s*([0-9][0-9,]{0,10})\s*\}')
        self.final_answer_pattern = re.compile(r'final\s+answer(?:\s+is)?\s*([0-9][0-9,]{0,10})', re.IGNORECASE)

    
    def _preload_model_weights(self) -> None:
        print(f'### Loading model weights from {self.cfg.model_path} into OS Page Cache...', flush=True)
        start_time = time.time()
    
        files_to_load = []
        total_size = 0
        
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
    
        def _read_file(path: str) -> None:
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
    
        with ThreadPoolExecutor(max_workers=16) as executor:
            list(executor.map(_read_file, files_to_load))
    
        elapsed = time.time() - start_time
        print(f'### Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n', flush=True)

    
    def _start_server(self) -> subprocess.Popen:
        cmd = [sys.executable]
        cmd.append('-m')
        cmd.append('vllm.entrypoints.openai.api_server')
        cmd.extend(
            [
                '--seed', 
                str(self.cfg.seed), 
                '--model', 
                self.cfg.model_path, 
                '--served-model-name', 
                self.cfg.served_model_name, 
                '--tensor-parallel-size', 
                '1', 
                '--max-num-seqs', 
                str(self.max_num_seqs), 
                '--max-num-batched-tokens', 
                str(self.cfg.max_num_batched_tokens),
                '--max-cudagraph-capture-size',
                str(self.cfg.max_cudagraph_capture_size),
                '--gpu-memory-utilization', 
                str(self.cfg.gpu_memory_utilization), 
                '--host', 
                '0.0.0.0', 
                '--port', 
                str(self.port), 
                '--dtype', 
                self.cfg.dtype, 
                '--kv-cache-dtype', 
                self.cfg.kv_cache_dtype, 
                '--max-model-len', 
                str(self.cfg.context_tokens), 
                '--stream-interval', 
                str(self.cfg.stream_interval), 
                '--async-scheduling', 
                '--disable-log-stats', 
                '--enable-prefix-caching',
                '--enable-chunked-prefill',
            ]
        )

        # When chache is quantized
        if self.cfg.calculate_kv_scales:
            cmd.append('--calculate-kv-scales')
        
        self.log_file = open('vllm_server.log', 'w')
        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )

        
    def _wait_for_server(self):
        print(f"### Waiting for vLLM server...", flush=True)
    
        start_time = time.time()
        timeout = float(self.cfg.server_timeout)
    
        while time.time() - start_time < timeout:
            return_code = self.server_process.poll()
    
            if return_code is not None:
                self.log_file.flush()
    
                with open("vllm_server.log", "r") as log_file:
                    logs = log_file.read()
    
                raise RuntimeError(f"Server died with code {return_code}. Full logs:\n{logs}\n")
    
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f"### Server is ready (took {elapsed:.2f} seconds).\n", flush=True)
                return
                
            except Exception:
                time.sleep(1)
    
        raise RuntimeError(f"Server failed to start (timeout).\n")

    
    def _initialize_kernels(self) -> None:        
        print(f'### Initializing {self.cfg.workers} persistent Jupyter kernels...', flush=True)
        start_time = time.time()
    
        self.sandbox_pool = queue.Queue()
    
        def _create_sandbox():
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
    
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
    
        elapsed = time.time() - start_time
        print(f'### Kernels initialized in {elapsed:.2f} seconds.\n', flush=True)

        
    def _scan_for_answer(self, text: str) -> int | None:
        matches = self.boxed_pattern.findall(text)
        
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
                
                if 0 <= value <= 99999:
                    return value
                    
            except ValueError:
                pass
                
        matches = self.final_answer_pattern.findall(text)
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
                
                if 0 <= value <= 99999:
                    return value
                    
            except ValueError:
                pass
                
        return None


    def _compute_windowed_entropy(self, logprobs_buffer: list, window: int = 200) -> float:
        if not logprobs_buffer:
            return float('inf')
    
        window = min(window, len(logprobs_buffer))
        relevant = logprobs_buffer[-window: ]
        
        if not relevant:
            return float('inf')
            
        total_entropy = 0.0
        count = 0
        
        for top_logprobs_dict in relevant:
            if not isinstance(top_logprobs_dict, dict) or not top_logprobs_dict:
                continue
                
            token_entropy = sum(
                -math.exp(lp) * math.log2(math.exp(lp)) 
                for lp in top_logprobs_dict.values()
                if math.exp(lp) > 0
            )
            
            total_entropy += token_entropy
            count += 1
        
        return total_entropy / count if count > 0 else float('inf')

    
    def _compute_mean_entropy(self, logprobs_buffer: list) -> float:
        if not logprobs_buffer:
            return float('inf')
    
        total_entropy = 0.0
        token_count = 0
    
        for top_logprobs_dict in logprobs_buffer:
            if not isinstance(top_logprobs_dict, dict):
                continue
                
            if not top_logprobs_dict:
                continue
            
            token_entropy = 0.0
            
            for token_str, log_prob in top_logprobs_dict.items():
                prob = math.exp(log_prob)
                
                if prob > 0:
                    token_entropy -= prob * math.log2(prob)
            
            total_entropy += token_entropy
            token_count += 1
    
        if token_count == 0:
            return float('inf')
    
        return total_entropy / token_count

    
    def _select_answer_v1(self, detailed_results: list) -> int:
        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)

        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']
            
            if answer is not None:
                weight = 1.0 / max(entropy, 1e-9)
                answer_weights[answer] += weight
                answer_votes[answer] += 1

        scored_answers = []
        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer, 
                'votes': answer_votes[answer], 
                'score': total_weight
            })

        scored_answers.sort(
            key=lambda x: (x['score'], x['votes']), 
            reverse=True
        )

        vote_data = [(item['answer'], item['votes'], item['score']) for item in scored_answers]
        vote_dataframe = pd.DataFrame(vote_data, columns=['Answer', 'Votes', 'Score'])
        vote_dataframe = vote_dataframe.round({'Score': 3})
        display(vote_dataframe)
        
        if not scored_answers:
            print(f'\nFinal Answer: 0\n', flush=True)
            return 0
    
        final_answer = scored_answers[0]['answer']    
        print(f'\nFinal Answer: {final_answer}\n', flush=True)
        return final_answer

    def _select_answer_v2(self, detailed_results: list) -> int:
        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)
    
        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']
            python_calls = result.get('Python Calls', 0)
            python_errors = result.get('Python Errors', 0)
    
            if answer is not None:
                confidence = 1.0 / max(entropy, 1e-9)
    
                if python_calls > 0:
                    python_quality = 1.0 - (python_errors / python_calls)
                    confidence *= (1.0 + 0.5 * python_quality)
    
                answer_weights[answer] += confidence
                answer_votes[answer] += 1
    
        scored_answers = []
        for answer, total_weight in answer_weights.items():
            votes = answer_votes[answer]
            hybrid_score = total_weight * 0.5 + votes * 0.5
            
            scored_answers.append({
                'answer': answer,
                'votes': votes,
                'confidence': round(total_weight, 3),
                'score': hybrid_score,
            })
    
        scored_answers.sort(
            key=lambda x: (x['score'], x['votes']),
            reverse=True
        )
    
        vote_data = [(item['answer'], item['votes'], item['confidence'], item['score']) for item in scored_answers]
        vote_dataframe = pd.DataFrame(vote_data, columns=['Answer', 'Votes', 'Confidence', 'Score'])
        vote_dataframe = vote_dataframe.round({'Confidence': 3, 'Score': 3})
        display(vote_dataframe)
    
        if not scored_answers:
            print(f'\nFinal Answer: 0\n', flush=True)
            return 0
    
        final_answer = scored_answers[0]['answer']
        print(f'\nFinal Answer: {final_answer}\n', flush=True)
        return final_answer


    def _stream_completion(
        self, 
        prompt_ids: list, 
        temperature: float,
        attempt_seed: int, 
        logprobs_buffer: list, 
        stop_event: threading.Event = None, 
        deadline: float = 0,
    ) -> tuple[list[int], list[str], int | None]:
    
        max_tokens = self.cfg.context_tokens - len(prompt_ids)
        
        if max_tokens < self.cfg.buffer_tokens:
            return [], [], None
            
        stream = self.client.completions.create(
            model=self.cfg.served_model_name, 
            temperature=temperature,
            logprobs=self.cfg.top_logprobs, 
            max_tokens=max_tokens, 
            prompt=prompt_ids, 
            seed=attempt_seed, 
            stream=True, 
            extra_body=self.extra_body,
        )
        
        try:
            token_buffer = []
            text_chunks = []
            final_answer = None
            
            for chunk in stream:
                if (stop_event is not None and stop_event.is_set()) or time.time() > deadline:
                    break
                    
                new_tokens = chunk.choices[0].token_ids
                new_text = chunk.choices[0].text
                
                if new_tokens:
                    token_buffer.extend(new_tokens)
                    text_chunks.append(new_text)
                    
                    chunk_logprobs = chunk.choices[0].logprobs
                    if chunk_logprobs is not None:
                        if chunk_logprobs.top_logprobs:
                            logprobs_buffer.extend(chunk_logprobs.top_logprobs)
                            
                if '}' in new_text:
                    search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                    answer = self._scan_for_answer(search_text)
                    
                    if answer is not None:
                        final_answer = answer
                        break
                        
        finally:
            if stream is not None:
                try:
                    stream.close()
                except Exception:
                    pass
            
        return token_buffer, text_chunks, final_answer


    def _handle_tool_call(
        self, 
        last_message: Message, 
        local_tool: AIMO3Tool
    ) -> tuple[int, list[Message]]:
    
        tool_responses = local_tool.process_sync_plus(last_message)
        response_text = tool_responses[0].content[0].text
        
        python_errors = 0
        if response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text:
            python_errors = 1            
        return python_errors, tool_responses

        
    def _process_attempt(
        self, 
        problem: str, 
        system_prompt: str, 
        temperature: float,
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float
    ) -> dict:
    
        start_time = time.time()
        
        if stop_event.is_set() or time.time() > deadline:
            print(f"- Attempt index {attempt_index} | Deadline reached or stop event set", flush=True)
            return {
                'Attempt': attempt_index, 
                'Answer': None, 
                'Python Calls': 0, 
                'Python Errors': 0, 
                'Response Length': 0, 
                'Entropy': float('inf'), 
                'Time': '00:00'
            }
            
        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        
        logprobs_buffer = []
        
        attempt_seed = random.getrandbits(63)
        
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
            
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox
            )
            
            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt, 
                problem, 
                local_tool.tool_config
            )
            
            conversation = Conversation.from_messages(messages)
            
            for i in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    print(f"- Attempt index {attempt_index} | Deadline reached or stop event set", flush=True)
                    break
                    
                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                
                token_buffer, text_chunks, answer = self._stream_completion(
                    prompt_ids,
                    temperature,
                    attempt_seed, 
                    logprobs_buffer, 
                    stop_event, 
                    deadline
                )
                
                total_tokens += len(token_buffer)
                
                if answer is not None:
                    final_answer = answer
                    print(f"- Attempt index {attempt_index} | An answer available: {final_answer}", flush=True)
                    break
                    
                if not token_buffer:
                    print(f"- Attempt index {attempt_index} | No tokens produced", flush=True)
                    break
                    
                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]
                
                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    final_answer = self._scan_for_answer(answer_text)
                    if final_answer is not None:
                        print(f"- Attempt index {attempt_index} | An answer available: {final_answer}", flush=True)
                    break
                    
                if last_message.recipient == 'python':
                    python_calls += 1
                    errors, tool_responses = self._handle_tool_call(last_message, local_tool)
                    python_errors += errors
                    
                    conversation.messages.extend(tool_responses)
                    
        except Exception as exc:
            exc_type, exc_value, exc_traceback = sys.exc_info()
            print(f"- Attempt index {attempt_index} | Python error: {exc_type} - {exc_value} - {exc_traceback}", flush=True)
            python_errors += 1
            
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)
                
        entropy = self._compute_windowed_entropy(logprobs_buffer, window=self.cfg.window_entropy)
        
        elapsed = time.time() - start_time
        time_duration = f'{int(elapsed // 60):02d}:{int(elapsed % 60):02d}'
        
        return {
            'Attempt': attempt_index, 
            'Response Length': total_tokens, 
            'Python Calls': python_calls, 
            'Python Errors': python_errors, 
            'Entropy': entropy, 
            'Time': time_duration, 
            'Answer': final_answer
        }


    def process_attempt_loop(
        self,
        attempts: int,
        problem: str,
        system_prompt: str,
        temperatures: list[float],
        workers: int,
        early_stop: int,
        deadline: float
    ):
        
        tasks = [(system_prompt, i) for i in range(attempts)]
            
        stop_event = threading.Event()
        executor = ThreadPoolExecutor(max_workers=workers)

        detailed_results = []
        valid_answers = []
        futures = []
        
        try:
            for i, (system_prompt, attempt_index) in enumerate(tasks):
                future = executor.submit(
                    self._process_attempt, 
                    problem, 
                    system_prompt, 
                    temperatures[i],
                    attempt_index, 
                    stop_event, 
                    deadline
                )
                
                futures.append(future)
                
            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
                    
                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])

                    if early_stop:
                        counts = Counter(valid_answers).most_common(1)
                        if counts and counts[0][1] >= early_stop:
                            stop_event.set()
                            
                            for f in futures:
                                f.cancel()
                            break
                        
                except Exception as exc:
                    print(f'### Future failed: {exc}')
                    continue
        finally:
            if not stop_event.is_set():
                stop_event.set()
            try:
                executor.shutdown(wait=True, cancel_futures=True)
            except TypeError:
                executor.shutdown(wait=True)
    
            time.sleep(0.02)

        return detailed_results, valid_answers


    def _compute_times(self):
        start_time = time.time()
    
        elapsed_global = start_time - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left = max(1, self.problems_remaining)
        fair_share = time_left / problems_left
        min_budget = max(self.cfg.min_absolute_timeout, fair_share * 0.6)
        budget = min(fair_share * 4, self.cfg.max_absolute_timeout)
        budget = max(budget, min_budget)
        
        deadline = time.time() + budget
        print(f'### Time left: {time_left:.1f}s | Fair share: {fair_share:.1f}s | Budget: {budget:.1f} | Deadline: {deadline:.2f}\n', flush=True)

        return start_time, deadline

    
    def _should_stop_after_phase_1(self, results: list, entropy_threshold: float = 0.90) -> tuple[bool, str]:
        valid = [r for r in results if r['Answer'] is not None]
    
        if not valid:
            return False, f"No answers in phase 1"
    
        answers = [r['Answer'] for r in valid]
        
        if len(set(answers)) == 1:
            mean_entropy = sum(r['Entropy'] for r in valid) / len(valid)
            if mean_entropy > entropy_threshold:
                return False, f"Full consensus but high entropy ({mean_entropy:.3f} > {entropy_threshold})"
            return True, f"Strong full consensus: answer={answers[0]}, entropy={mean_entropy:.3f}"

        if len(valid) > 2:
            counts = Counter(answers)
            most_common_answer, most_common_count = counts.most_common(1)[0]
            
            if most_common_count > len(valid) / 2:
                majority_results = [r for r in valid if r['Answer'] == most_common_answer]
                mean_entropy = sum(r['Entropy'] for r in majority_results) / len(majority_results)
                
                if mean_entropy > entropy_threshold:
                    return False, f"Majority consensus ({most_common_count}/{len(valid)}) but high entropy ({mean_entropy:.3f} > {entropy_threshold})"
                return True, f"Strong majority consensus: answer={most_common_answer} ({most_common_count}/{len(valid)}), entropy={mean_entropy:.3f}"
        
            return False, f"No majority: {dict(counts)}"

        return False, f"Disagreement with insufficient attempts: {answers}"
            


    def solve_problem(self, problem: str) -> int:    
        start_time, deadline = self._compute_times()  

        print(f"### Phase 1 start")
        detailed_results, valid_answers = self.process_attempt_loop(
            attempts=self.phase_1_attempts,
            problem=problem,
            system_prompt=self.cfg.system_prompt,
            temperatures=self.temperatures_phase_1,
            workers=self.cfg.workers,
            early_stop=self.cfg.early_stop_phase_1,
            deadline=deadline,
        )

        stop, ret = self._should_stop_after_phase_1(detailed_results, self.cfg.entropy_threshold)
        print(f"### Phase 1 should stop: {ret}\n")
        
        if not stop:
            print(f"### Phase 2 start")
            detailed_results_phase_2, valid_answers_phase_2 = self.process_attempt_loop(
                attempts=self.phase_2_attempts,
                problem=problem,
                system_prompt=self.cfg.system_prompt + "\n" + self.cfg.phase_2_extra_prompt,
                temperatures=self.temperatures_phase_2,
                workers=self.cfg.workers,
                early_stop=self.cfg.early_stop_phase_2,
                deadline=deadline,
            )

            detailed_results.extend(detailed_results_phase_2)
            valid_answers.extend(valid_answers_phase_2)
        
        if detailed_results:
            results_dataframe = pd.DataFrame(detailed_results)
            results_dataframe['Entropy'] = results_dataframe['Entropy'].round(3)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64', copy=False)
            display(results_dataframe)

        self.problems_remaining = max(0, self.problems_remaining - 1)

        elapsed = time.time() - start_time
        time_duration = f'{int(elapsed // 60):02d}:{int(elapsed % 60):02d}'
        print(f'\n### Time elaped: {time_duration}')
            
        if not valid_answers:
            print(f'\nResult: 0\n')
            return 0

        final_answer = self._select_answer_v2(detailed_results)

        return final_answer


    def __del__(self):
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
            
        if hasattr(self, 'log_file'):
            self.log_file.close()
            
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
                    
                except Exception:
                    pass

In [13]:
solver = AIMO3Solver(CFG)

### Loading model weights from /kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1 into OS Page Cache...
### Processed 26 files (65.28 GB) in 61.67 seconds.

### Waiting for vLLM server...
### Server is ready (took 136.30 seconds).

### Initializing 8 persistent Jupyter kernels...
### Kernels initialized in 1.69 seconds.



In [14]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    id_value = id_.item(0)
    question_text = question.item(0)
    
    print(f'\n=============================================================== ID: {id_value} ===============================================================', flush=True)
    
    try:
        if answer is not None:
            print(f"### Ground Truth: {answer.item(0)}\n", flush=True)
    except Exception as e:
        pass
    
    print(f'### Problem: {question_text}\n', flush=True)

    gc.disable()
    
    final_answer = solver.solve_problem(question_text)

    try:
        if answer is not None:
            print(f"\n### Final result | Ground Truth: {answer.item(0)} | Predicted: {final_answer}", flush=True)
    except Exception as e:
        pass
        
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [15]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    solver.problems_remaining = 50
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv',)
        #('/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv',)
    )


=============================================================== ID: 111bbb ===============================================================
### Problem: What is $0\times10$?

### Time left: 17524.6s | Fair share: 350.5s | Budget: 680.0 | Deadline: 1772406346.18

### Phase 1 start
- Attempt index 2 | An answer available: 0
- Attempt index 0 | An answer available: 0
- Attempt index 1 | An answer available: 0
### Phase 1 should stop: Strong full consensus: answer=0, entropy=0.681



,Attempt,Response Length,Python Calls,Python Errors,Entropy,Time,Answer
0,2,51,0,0,0.802,00:13,0
1,0,101,0,0,0.672,00:14,0
2,1,101,0,0,0.569,00:14,0



### Time elaped: 00:14


,Answer,Votes,Confidence,Score
0,0,3,4.493,3.747



Final Answer: 0


=============================================================== ID: 000aaa ===============================================================
### Problem: What is $1-1$?

### Time left: 17510.2s | Fair share: 357.4s | Budget: 680.0 | Deadline: 1772406360.53

### Phase 1 start
- Attempt index 2 | An answer available: 0
- Attempt index 1 | An answer available: 0
- Attempt index 0 | An answer available: 0
### Phase 1 should stop: Strong full consensus: answer=0, entropy=0.549



...


Final Answer: 0


=============================================================== ID: 222ccc ===============================================================
### Problem: Solve $4+x=4$ for $x$.

### Time left: 17509.4s | Fair share: 364.8s | Budget: 680.0 | Deadline: 1772406361.32

### Phase 1 start
- Attempt index 0 | An answer available: 0
- Attempt index 1 | An answer available: 0
- Attempt index 2 | An answer available: 0
### Phase 1 should stop: Strong full consensus: answer=0, entropy=0.704



,Attempt,Response Length,Python Calls,Python Errors,Entropy,Time,Answer
0,0,92,0,0,0.632,00:00,0
1,1,101,0,0,1.013,00:00,0
2,2,106,0,0,0.468,00:00,0



### Time elaped: 00:00


,Answer,Votes,Confidence,Score
0,0,3,4.708,3.854



Final Answer: 0

